# Day 5.3 — Tool Registry and Discovery — You Build It
On Day 1 a tool lived in a dictionary and `execute_tool_call` ran whatever it found there. That has
nowhere to record *how dangerous* a tool is, and no way to give two agents different tools. This is
the day's one hands-on exercise: write the registry, check it, then read the reference — which the
rest of the day uses.

### Where Day 1 stops

Run the Day 1 arrangement once more and read what it cannot express.

In [ ]:
print("Day 1 tool dispatch:")
print("  ", execute_tool_call({"id": "1", "name": "lookup_notes", "arguments": {"query": "mcp"}}, DAY1_TOOLS))
print()
print("Keys stored per Day 1 tool:", sorted(DAY1_TOOLS["lookup_notes"]))
print()
print("What that arrangement cannot answer:")
print("  - which of these tools may THIS agent see?       one global dict, so: all of them")
print("  - how risky is this tool?                        there is no field for it")
print("  - who may run it, and when?                      nothing checks")
print()
print("Three separate questions are hiding in there, and today they get three separate answers:")
print("  discovery  : does this agent get told the tool exists?     (registry, this section)")
print("  validation : are these arguments structurally acceptable?  (registry, this section)")
print("  policy     : may this agent perform this action right now? (5.4)")

### The contract

```python
registry.register(spec, function)          # duplicate name -> ValueError
registry.discover(allowed=None)            # -> list[ToolSpec]; allowed=None means everything
registry.call(name, arguments, allowed)    # unknown -> KeyError; ungranted -> PermissionError;
                                           # bad arguments -> ValueError/TypeError; else run it
```

`get` and `validate` are written for you: `get` raises `KeyError` for an unregistered name, and
`validate` checks the argument shape. Your `call` uses both, refuses an ungranted tool, and only then
executes. Filtering `discover` hides a tool from the model; the check inside `call` is what actually
stops it running.

In [ ]:
@dataclass
class RegisteredTool:
    spec: ToolSpec
    function: Callable

class ToolRegistry:
    """YOUR TURN. Fill in register, discover and call."""

    def __init__(self):
        self._tools = {}                       # name -> RegisteredTool, in registration order

    def register(self, spec, function):
        """Store spec + function. Registering the same name twice raises ValueError."""
        raise NotImplementedError("write register")

    def discover(self, allowed=None):
        """Return the ToolSpecs this agent may see. allowed=None means everything registered."""
        raise NotImplementedError("write discover")

    def call(self, name, arguments, allowed=None):
        """get() the tool, refuse it with PermissionError if it is not in `allowed`,
        validate() the arguments, and only then run the function."""
        raise NotImplementedError("write call")

    # --- written for you ---------------------------------------------------------
    def get(self, name):
        if name not in self._tools:
            raise KeyError(f"Unknown tool: {name}")
        return self._tools[name]

    def validate(self, name, arguments):
        """Structural check: required keys present, declared types respected, nothing undeclared."""
        schema = self.get(name).spec.input_schema
        missing = [key for key in schema.get("required", []) if key not in arguments]
        if missing:
            raise ValueError(f"Missing required arguments: {missing}")
        expected = {"string": str, "integer": int, "number": (int, float),
                    "boolean": bool, "array": list, "object": dict}
        for key, value in arguments.items():
            prop = schema.get("properties", {}).get(key)
            if prop is None and schema.get("additionalProperties") is False:
                raise ValueError(f"Unexpected argument: {key}")
            if prop and prop.get("type") in expected and not isinstance(value, expected[prop["type"]]):
                raise TypeError(f"{key} must be {prop['type']}")

print("ToolRegistry stub defined - fill in the three methods, then run the check below")

### Behavioural check

Two registrations and seven calls with known right answers. It prints a hint instead of failing
while your registry is unfinished.

In [ ]:
NOTES_TOOL = (ToolSpec("lookup_notes", "Look up a course note",
                       {"type": "object", "properties": {"query": {"type": "string"}},
                        "required": ["query"], "additionalProperties": False}, "read"),
              lambda query: {"query": query, "note": COURSE_NOTES.get(query, "no supplied note")})
DRAFT_TOOL = (ToolSpec("create_draft", "Create a reversible local draft",
                       {"type": "object", "properties": {"subject": {"type": "string"}, "body": {"type": "string"}},
                        "required": ["subject", "body"], "additionalProperties": False}, "write"),
              lambda subject, body: {"draft": True, "subject": subject, "body": body})

def run_registry_checks():
    registry = ToolRegistry()
    registry.register(*NOTES_TOOL)
    registry.register(*DRAFT_TOOL)

    # Discovery is scoped, and an empty allow-list really means nothing.
    assert [s.name for s in registry.discover()] == ["lookup_notes", "create_draft"]
    assert [s.name for s in registry.discover(["lookup_notes"])] == ["lookup_notes"]
    assert registry.discover([]) == []
    print("discovery  : all =", [s.name for s in registry.discover()],
          "| scoped =", [s.name for s in registry.discover(["lookup_notes"])])

    # A permitted, well-formed call runs the function.
    assert registry.call("lookup_notes", {"query": "harness"}, ["lookup_notes"])["query"] == "harness"
    print("valid call :", registry.call("lookup_notes", {"query": "harness"}, ["lookup_notes"]))

    for label, attempt, expected in [
        ("duplicate registration", lambda: registry.register(*NOTES_TOOL), ValueError),
        ("unknown tool", lambda: registry.call("erase_disk", {}, ["lookup_notes"]), KeyError),
        ("ungranted tool", lambda: registry.call("create_draft", {"subject": "s", "body": "b"}, ["lookup_notes"]), PermissionError),
        ("missing argument", lambda: registry.call("create_draft", {"subject": "s"}, ["create_draft"]), ValueError),
        ("wrong type", lambda: registry.call("create_draft", {"subject": "s", "body": 42}, ["create_draft"]), TypeError),
        ("undeclared argument", lambda: registry.call("lookup_notes", {"query": "x", "extra": 1}, ["lookup_notes"]), ValueError),
    ]:
        try:
            attempt()
        except expected as exc:
            print(f"{label:<23} -> {type(exc).__name__}: {exc}")
        else:
            raise AssertionError(f"{label} must raise {expected.__name__}")
    print("PASS: registration, scoped discovery, permission and validation all behave correctly")

try:
    run_registry_checks()
except NotImplementedError:
    print("Not implemented yet. Fill in ToolRegistry above, or read the reference solution below and re-run this cell.")

### Reference solution

Read it line by line, then re-run the check cell. **This is the registry the rest of the day uses.**

In [ ]:
# --- Reference solution ------------------------------------------------------------
class ToolRegistry:
    def __init__(self):
        self._tools = {}

    def register(self, spec, function):
        if spec.name in self._tools:                       # one name, one implementation
            raise ValueError(f"Duplicate tool: {spec.name}")
        self._tools[spec.name] = RegisteredTool(spec, function)

    def discover(self, allowed=None):
        # Least privilege: an agent is never even TOLD about a tool it may not use.
        names = set(allowed) if allowed is not None else set(self._tools)
        return [tool.spec for name, tool in self._tools.items() if name in names]

    def call(self, name, arguments, allowed=None):
        tool = self.get(name)                              # 1. unknown name -> KeyError
        if allowed is not None and name not in allowed:    # 2. second line of defence: it stops a
            raise PermissionError(f"Tool {name!r} is not on this agent's allow-list")  # named-but-hidden tool
        self.validate(name, arguments)                     # 3. shape, before any side effect
        return tool.function(**arguments)                  # 4. only now does the host run it

    def get(self, name):
        if name not in self._tools:
            raise KeyError(f"Unknown tool: {name}")
        return self._tools[name]

    def validate(self, name, arguments):
        schema = self.get(name).spec.input_schema
        missing = [key for key in schema.get("required", []) if key not in arguments]
        if missing:
            raise ValueError(f"Missing required arguments: {missing}")
        expected = {"string": str, "integer": int, "number": (int, float),
                    "boolean": bool, "array": list, "object": dict}
        for key, value in arguments.items():
            prop = schema.get("properties", {}).get(key)
            if prop is None and schema.get("additionalProperties") is False:
                raise ValueError(f"Unexpected argument: {key}")
            if prop and prop.get("type") in expected and not isinstance(value, expected[prop["type"]]):
                raise TypeError(f"{key} must be {prop['type']}")

run_registry_checks()

### Four tools, four risk levels

The registry is also where a tool's **risk** is written down: a person's judgement about
consequences, stored next to the schema.

In [ ]:
def build_demo_registry():
    """The four tools used from here to 5.7, deliberately spanning the four risk levels."""
    registry = ToolRegistry()
    registry.register(*NOTES_TOOL)                                            # read
    registry.register(*DRAFT_TOOL)                                            # write
    registry.register(ToolSpec("send_email", "Send a simulated external email",
                               {"type": "object",
                                "properties": {"to": {"type": "string"}, "subject": {"type": "string"},
                                               "body": {"type": "string"}},
                                "required": ["to", "subject", "body"], "additionalProperties": False},
                               "external"),
                      lambda to, subject, body: {"sent": True, "to": to, "subject": subject, "body": body})
    registry.register(ToolSpec("erase_workspace", "Delete all workspace data",
                               {"type": "object", "properties": {}, "additionalProperties": False},
                               "destructive"),
                      lambda: {"erased": True})
    return registry

registry = build_demo_registry()
research, task = load_config("research_agent"), load_config("task_agent")

print(f"{'tool':<17}{'risk':<13}required arguments")
print("-" * 62)
for spec in registry.discover():
    print(f"{spec.name:<17}{spec.risk:<13}{', '.join(spec.input_schema.get('required', [])) or '(none)'}")

print()
print("Registered in total   :", [s.name for s in registry.discover()])
print("research_agent may see:", [s.name for s in registry.discover(research.allowed_tools)])
print("task_agent may see    :", [s.name for s in registry.discover(task.allowed_tools)])
print()
print("erase_workspace is registered and appears in neither list. Not mentioning a tool is the")
print("cheapest safety control there is - and 5.4 adds the one that still works when a model")
print("names a tool nobody offered it.")

### Checkpoint

**1. The arguments validated and the tool was on the allow-list. Why is a policy still needed?**

<details><summary>Show answer</summary>

Validation only answers *is this well-formed?*. A well-formed `send_email` may still be something this agent must not do unsupervised.

</details>

**2. Why does `discover` filter *and* `call` check the allow-list again?**

<details><summary>Show answer</summary>

Filtering removes temptation. But a model can still name a hidden tool, and other code calls `call` directly, so only the check inside `call` prevents execution.

</details>

### Recap

- **Limitation seen:** a global tool dictionary cannot scope tools per agent or record risk.
- **Layer added:** a registry with scoped discovery, validation and a permission check at dispatch.
- **Evidence:** six malformed or unauthorised calls refused before any function body ran.